# Traditional ML with Amazon SageMaker AI 

## Workshop Overview

Welcome to this hands-on workshop on building traditional machine learning models with Amazon SageMaker! In this workshop, you'll learn how to:

- **Set up** your SageMaker environment with the latest Python SDK v3
- **Configure** MLflow for experiment tracking and model management
- **Prepare** and process data for machine learning
- **Train** an XGBoost model using SageMaker's ModelTrainer
- **Package** a model with a custom inference handler, and **register** it with a frozen drift baseline
- **Deploy** models to real-time endpoints
- **Test** and validate your deployed model
- **Clean up** resources to avoid unnecessary costs

### Business Problem for the workshop

You'll work with a **bank marketing dataset** to predict whether a customer will subscribe to a term deposit based on:
- Demographics (age, job, marital status, education)
- Financial information (credit default, housing loan, personal loan)
- Campaign data (contact type, month, day of week, duration)
- Economic indicators (employment rate, consumer price index, etc.)

This is a **binary classification** problem commonly faced by financial institutions and we will use this as an example to run the Traditional ML with Amazon SageMaker AI  workshop.

### Dataset Information

- **Source**: UCI Machine Learning Repository - Bank Marketing Dataset. 
- **Size**: ~41,000 records with 20 features
- **Target**: Binary (yes/no) - Will the customer subscribe?
- **Citation**: [Moro et al., 2014] S. Moro, P. Cortez and P. Rita. A Data-Driven Approach to Predict the Success of Bank Telemarketing.

### AWS Services Used

- **Amazon SageMaker AI Training**: Managed training infrastructure
- **Amazon SageMaker Endpoints**: Real-time model hosting
- **Amazon SageMaker AI MLflow**: Experiment tracking and model registry
- **Amazon S3**: Data and model artifact storage
- **AWS IAM**: Security and access management
- **SageMaker Python SDK v3**

### Estimated Time

- **Total**: 45-60 minutes
- **Training**: ~5 minutes
- **Deployment**: ~5-7 minutes

### Prerequisites

- AWS account with SageMaker Studio access

---
## How This Notebook Fits Together

Seven sections, each handing something specific to the next:

| Section | Produces | Consumed by |
|---|---|---|
| 1. Setup | `sagemaker_session`, `region`, `role`, `bucket` | everything below |
| 2. MLflow | `mlflow_app_arn`, `mlflow_experiment_name` | the training job's environment |
| 3. Iceberg read | pinned snapshot IDs, `train.csv` / `test.csv`, `X_test` / `y_test` | Sections 4, 5, 6 |
| 4. Training | trained booster in S3, an MLflow run | Section 5 |
| 5. Register + deploy | `model_package_arn`, `baseline_s3_uri`, `endpoint_name` | Sections 6, 7, Lab 5 |
| 6. Test | live predictions, captured rows in Iceberg | Lab 5 |
| 7. Verify | proof Lab 5 can find this model's baseline | Lab 5 |

Two contracts run the full length of the notebook. Most of the care in the code
is spent protecting them.

**Feature order.** `FEATURES` in Section 3 must match the order Lab 2 wrote.
XGBoost trains on header-less CSV, so it knows columns by position only. A
column in the wrong slot does not raise anything -- it produces a quietly wrong
model.

**The baseline chain.** Section 5 records *which Iceberg snapshot the model
trained on* and attaches that record to the model package. Lab 5 walks backwards
from the running endpoint to find it. Section 7 proves the walk works before you
leave this notebook.

### If a cell fails partway through

State lives in the kernel, so cells are not independent. A failed cell usually
means: fix that one cell, run it, and carry on -- everything from earlier cells
is still in memory. Restarting the kernel, by contrast, means starting again
from Section 1.

Only Section 5's final cell creates something that bills by the hour (the
endpoint). The cleanup cell at the end removes it.

---

## Section 1: Prerequisites & Setup

### Key Concepts

**SageMaker Python SDK v3**: The latest version of the SageMaker SDK provides:
- Simplified APIs for training and deployment
- Better integration with MLflow
- Improved resource management
- Type hints and better IDE support
- See [documentation](https://sagemaker.readthedocs.io/en/stable/) for more


### Instructions

Run the cells below to set up your environment. The first cell may take 1-2 minutes to install packages.

### Step 1.1: Install pinned dependencies

**Expect the kernel to restart.** The last line of this cell calls
`do_shutdown(restart=True)` on purpose, so Python picks up the freshly installed
packages. Studio will report the kernel restarting or disconnecting -- that is
success, not a crash.

Versions are pinned deliberately:

- `sagemaker>=3.3.1,<4` for SDK v3 (`ModelTrainer`, and the typed `Model` /
  `EndpointConfig` / `Endpoint` resources used to deploy in Section 5)
- `mlflow>=3.4,<4` -- MLflow 3 is what introduces **logged models** as
  first-class entities with their own `model_id`, which Section 5 registers
- `sagemaker-mlflow>=0.5,<1` -- `log_inference_specification()`, the call that
  makes an MLflow model deployable, only exists from 0.5.0 on
- `xgboost>=3,<4` -- matches the XGBoost 3.x training and inference images;
  older clients can load the newer JSON without raising but return incorrect
  predictions, so this is a correctness requirement rather than convenience

**Run this cell once**, wait for the restart, then continue with the next cell.
Nothing is defined yet, so the restart costs you nothing.


In [ ]:
# Install required packages
# This may take 1-2 minutes on first run
# Ignore dependency conflicts warnings and errors.
!pip install --upgrade pip -q

# sagemaker-mlflow >= 0.5.0 is a hard requirement of Section 5:
# log_inference_specification() does not exist in earlier releases, and mlflow 3
# is what gives us logged models (model_id) to register.
%pip install --no-cache-dir -q "sagemaker>=3.3.1,<4" "sagemaker-train>=1,<2" \
    "sagemaker-serve>=1,<2" "mlflow>=3.4,<4" "sagemaker-mlflow>=0.5,<1" \
    "pandas" "scikit-learn" "xgboost>=3,<4" "awswrangler>=3.9"

# Restart kernel to pick up updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


### Step 1.2: Imports and session

The next two cells set up everything the rest of the notebook depends on.

The SDK v3 imports are worth knowing by name, because they replace the v2
`Estimator` and `Model.deploy()`:

- `ModelTrainer` -- describes and launches a training job
- `SourceCode`, `Compute`, `InputData` -- the job's script, hardware, and data channels
- `Model`, `EndpointConfig`, `Endpoint` + `ContainerDefinition`, `ProductionVariant` --
  the typed resources Section 5 deploys with, one per SageMaker API object
- `Session`, `get_execution_role` -- your Studio identity and its default S3 bucket
- `image_uris` -- looks up AWS-managed container images (XGBoost, in our case)

On the MLflow side, `MlflowClient` is what Section 5 uses to attach files to a
logged model, and `sagemaker_mlflow` carries the
`log_inference_specification()` call that makes that model deployable.

Then `Session()` resolves your region, execution role, and the default bucket
(`sagemaker-{region}-{account}`). Everything this notebook writes lands under
the `bank-marketing-lab/` prefix in that bucket. Check the printed values look
right before moving on.


In [ ]:
# Import required libraries
import boto3
import sagemaker
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import importlib.util
import json
import zipfile
import os
import io
import tempfile
import time
import awswrangler as wr
from datetime import datetime

# MLflow: the client attaches artifacts to a logged model, and the SageMaker
# plugin turns that logged model into something the Model Registry can deploy.
from mlflow import MlflowClient
import sagemaker_mlflow

# SageMaker v3 imports
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import SourceCode, InputData, Compute
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

print('\u2713 All libraries imported successfully')
print(f'MLflow version: {mlflow.__version__}')
print(f'sagemaker-mlflow version: {sagemaker_mlflow.importlib_metadata.version("sagemaker-mlflow")}')


In [ ]:
# Initialize SageMaker session and get AWS configuration
import os, sys
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
role = get_execution_role()
bucket = sagemaker_session.default_bucket()

# Data prefix comes from the single shared source (repo-root .env via the
# shared loader) — lab2 writes here and lab3 reads it, so it must not drift.
for _c in [os.getcwd(), *[str(p) for p in __import__('pathlib').Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_c, 'workshop_env.py')):
        sys.path.insert(0, _c); break
from workshop_env import load_workshop_env
_env = load_workshop_env()
prefix = _env['DATA_PREFIX']

print('AWS Configuration:')
print(f'  Region: {region}')
print(f'  S3 Bucket: {bucket}')
print(f'  IAM Role: {role}')
print(f'  Data Prefix: {prefix}')
print('\n SageMaker session initialized successfully')

---

## Section 2: SageMakerAI MLflow App Configuration

### What You'll Learn

In this section, you'll:
1. Connect to your SageMaker AI MLflow app
2. Create or select an MLflow experiment
3. Understand MLflow's role in experiment tracking

### Key Concepts

**Amazon SageMaker AI MLflow App**: A fully managed SageMakerAI service that provides:
- **Experiment Tracking**: Log parameters, metrics, and artifacts
- **Model Registry**: Version and manage models. 
- **SageMakerAI integration**: Automatically, register models to SageMakerAI Model Registry
- **Reproducibility**: Track code versions and dependencies
- **Collaboration**: Share experiments across teams
- **Integration**: Works seamlessly with SageMaker training

**MLflow Experiment**: A logical grouping of runs that:
- Organizes related training attempts
- Enables comparison of different approaches
- Tracks the evolution of your model

**MLflow Run**: A single execution that logs:
- Hyperparameters (learning rate, tree depth, etc.)
- Metrics (accuracy, precision, recall, AUC)
- Artifacts (model files, plots, data samples)
- Metadata (start time, duration, user)

### Instructions

1. **Find your MLflow App ARN**:
   - In SageMaker Studio, navigate to the left sidebar
   - Click on "MLflow" palen under "Applications"
   - Find your MLflow app (usually named "DefaultMLFlowApp"). For the first time, it will take 1-2mins to launch
   - Copy the ARN if you need to use a specific app

### Step 2.1: Point at your MLflow app

This cell lists every MLflow app in the account, looks for the one named in
`mlflow_app_name`, and then checks one setting on it.

**You may need to edit that name.** If the cell raises
`ValueError: MLflow app "..." not found`, read the list it printed just above
the error, copy the name you actually have, and re-run. The app name varies with
how the workshop stack was deployed.

If the list is empty, the app has not been created yet -- open **Applications ->
MLflow** in the Studio sidebar and start it. A first launch takes 1-2 minutes.

The ARN this resolves into `mlflow_app_arn` is used twice: once locally as the
tracking URI, and once as an environment variable on the training job, so the
training container logs to the same place.

**The setting it checks is `ModelRegistrationMode`.** With
`AutoModelRegistrationEnabled`, every `mlflow.register_model()` call is mirrored
into the SageMaker AI Model Registry as a Model Package. Section 5 is built on
that mirror -- it registers once, in MLflow, and deploys the Model Package
SageMaker creates from it. The workshop stack sets this
(`templates/3-sagemaker.yaml`); the cell flips it for you if your app predates
that, which takes about a minute and needs no restart.


In [ ]:
# Configure MLflow app connection
# Single source of truth: PROJECT_NAME from the environment (set at deploy time),
# falling back to the one canonical default. The app name is derived from it.
import os
PROJECT_NAME = os.environ.get('PROJECT_NAME', 'bank-marketing-prediction')
mlflow_app_name = f'{PROJECT_NAME}-mlflow-app'

# Get MLflow app ARN
sm_client = boto3.client('sagemaker', region_name=region)
mlflow_list = sm_client.list_mlflow_apps()

print(f'Found {len(mlflow_list["Summaries"])} MLflow app(s) in your account:')
for app in mlflow_list['Summaries']:
    print(f'  - {app["Name"]}')

# Find the specified MLflow app
mlflow_app_arn = None
for mlflow_app in mlflow_list['Summaries']:
    if mlflow_app['Name'] == mlflow_app_name:
        mlflow_app_arn = mlflow_app['Arn']
        break

if mlflow_app_arn:
    print(f'\n Using MLflow app: {mlflow_app_name}')
    print(f'  ARN: {mlflow_app_arn}')
else:
    raise ValueError(f'MLflow app "{mlflow_app_name}" not found. Please check the name or create one in SageMaker Studio.')

# Section 5 registers the model ONCE, in MLflow, and deploys the Model Package
# that SageMaker mirrors from it. That mirror only runs when the app is in
# AutoModelRegistrationEnabled mode, so check it here rather than discovering it
# as a registration that never produces a package.
_app = sm_client.describe_mlflow_app(Arn=mlflow_app_arn)
print(f'  Model registration mode: {_app.get("ModelRegistrationMode", "(unset)")}')

if _app.get('ModelRegistrationMode') != 'AutoModelRegistrationEnabled':
    print('\n  Enabling auto model registration (MLflow -> SageMaker Model Registry)...')
    sm_client.update_mlflow_app(
        Arn=mlflow_app_arn, ModelRegistrationMode='AutoModelRegistrationEnabled'
    )
    # The update is asynchronous; the app keeps serving throughout.
    for _ in range(30):  # ~5 minutes
        _app = sm_client.describe_mlflow_app(Arn=mlflow_app_arn)
        if (_app.get('ModelRegistrationMode') == 'AutoModelRegistrationEnabled'
                and _app['Status'] in ('Created', 'Updated')):
            break
        print('.', end='', flush=True)
        time.sleep(10)
    else:
        raise TimeoutError(
            'MLflow app did not reach AutoModelRegistrationEnabled. Section 5 needs '
            'it; enable it on the app (Studio: Applications -> MLflow, or '
            'update-mlflow-app) before continuing.')
    print(f'\n  Auto model registration enabled (status {_app["Status"]}).')


### Step 2.2: Create or select the experiment

`set_tracking_uri(mlflow_app_arn)` points the MLflow client at the managed app
rather than a local `mlruns/` directory. From here, MLflow calls travel to
SageMaker.

The cell then tries to create the experiment `bank-marketing-prediction` and
falls back to selecting it if it already exists -- so re-running the notebook
adds runs to one experiment instead of scattering them. On a second pass through
the lab, "Using existing" is the expected message.

In [ ]:
# Set MLflow tracking URI and create/select experiment
mlflow.set_tracking_uri(mlflow_app_arn)
mlflow_experiment_name = PROJECT_NAME

try:
    # Try to create a new experiment
    experiment_id = mlflow.create_experiment(mlflow_experiment_name)
    print(f' Created new MLflow experiment: {mlflow_experiment_name}')
    print(f'  Experiment ID: {experiment_id}')
except:
    # Experiment already exists, set it as active
    mlflow.set_experiment(mlflow_experiment_name)
    experiment = mlflow.get_experiment_by_name(mlflow_experiment_name)
    print(f' Using existing MLflow experiment: {mlflow_experiment_name}')
    print(f'  Experiment ID: {experiment.experiment_id}')

print('\n You can view your experiments in the SageMaker Studio MLflow App UI')

---

## Section 3: Load Prepared Data from Iceberg

Data preparation runs in one of **Lab 2B-2E** (Bank Marketing) followed by the shared Iceberg registration step, which publishes the feature-ready dataset as **Iceberg tables** in the Glue/Athena catalog (`bank_marketing`).

> **Prerequisite**: Run one of `lab2-data-prep/lab-2b-traditional-ml-data-prep-processing-job.ipynb` (or 2C/2D/2E), then `lab2-data-prep/lab-2-traditional-ml-iceberg-registration.ipynb`, first. Together they create `training_data` and `evaluation_data`.

### Input contract (produced by Lab 2B-2E + registration)

- Iceberg table `bank_marketing.training_data`
- Iceberg table `bank_marketing.evaluation_data`

Schema: `client_id`, `contact_timestamp`, `subscribed` (target), 20 feature columns, plus `prediction` / `probability_positive` placeholders.

The XGBoost training container consumes CSV, so the cells below query the Iceberg tables and materialize the target-first, header-less `train.csv` / `test.csv` the training job reads — the Iceberg tables remain the source of truth.

### Why we pin a snapshot before reading

Both tables are **live**: Lab 2 `overwrite()`s them, so their contents change whenever it re-runs. That is a problem for Lab 5, which measures drift by comparing live traffic against the training distribution — if the baseline moves to match current data, drift trends toward zero and a false negative looks exactly like a healthy run.

Iceberg's answer is **time travel**. Every write creates an immutable snapshot you can pin a query to:

```sql
SELECT * FROM bank_marketing.training_data FOR VERSION AS OF 8374927349827
```

So this section resolves one snapshot ID per table, reads the training CSVs *through* those snapshots, and Section 5 records the IDs in `baseline.json`. The result is that "the data this model was trained on" stays reproducible for the life of the model.


### Step 3.1: Table names, feature order, query output

Configuration only -- no data is read yet. Three things get established:

**Where the data lives**: the Glue database `bank_marketing` and its two tables.

**`FEATURES`** -- the 20 feature columns *in the exact order Lab 2 wrote them*.
This list is the positional contract described at the top of the notebook. It is
reused four more times: to build the training CSV, to score the model locally,
to name features inside the inference handler, and to record the schema in
`baseline.json`. `FEATURE_SCHEMA_VERSION` travels with it so Lab 5 can tell
which schema a drift verdict was computed against.

**Where Athena writes results**: Athena stages every query result in S3, so the
cell reads the Glue database's own `LocationUri` and derives an
`athena-query-results/` prefix on that same bucket. That bucket (`db_bucket`)
matters later -- it is also where `baseline.json` goes, because it is the bucket
Lab 5's monitor role is allowed to read.

In [ ]:
# Prepared data is published by Lab 2B-2E + the Iceberg registration step in Glue/Athena.
import awswrangler as wr
import os, sys

# Schema identifiers come from the single shared source (repo-root .env written
# by lab0-setup), read via the one shared loader — no hardcoded literals.
for _c in [os.getcwd(), *[str(p) for p in __import__('pathlib').Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_c, 'workshop_env.py')):
        sys.path.insert(0, _c); break
from workshop_env import load_workshop_env
_env = load_workshop_env()
ATHENA_DATABASE = _env['ATHENA_DATABASE']
TRAINING_TABLE = _env['TRAINING_TABLE']
EVALUATION_TABLE = _env['EVALUATION_TABLE']

# Feature order MUST match Lab 2 (positional CSV contract for XGBoost).
# Feature list comes from the single shared schema (workshop_common reads
# lab5-monitoring/src/config/dataset_schema.yaml) — no hardcoded feature list.
from workshop_common import schema as _schema
FEATURES = _schema.feature_names()

# Bump whenever FEATURES changes. It travels with the model in baseline.json and
# Lab 5 tags every drift verdict with it, so you can tell which feature schema a
# given verdict was computed against.
FEATURE_SCHEMA_VERSION = 1

# Resolve an Athena query-results location on the Glue database's bucket.
glue = boto3.client('glue', region_name=region)
_db = glue.get_database(Name=ATHENA_DATABASE)['Database']
db_location = (_db.get('LocationUri') or f's3://{bucket}/{ATHENA_DATABASE}').rstrip('/')
db_bucket = db_location.split('/')[2]
athena_output = f's3://{db_bucket}/athena-query-results/'

print(f'Reading Iceberg tables from {ATHENA_DATABASE} (results -> {athena_output})')

### Step 3.2: Pin one snapshot per table

This is the cell that makes drift detection meaningful, so it is worth
understanding rather than just running.

Drift detection asks: *does live traffic still look like the data this model was
trained on?* Answering it requires the training data **as it was at training
time**. But `training_data` and `evaluation_data` are live tables that Lab 2
overwrites on every run. Compare against "whatever is in the table now" and the
baseline moves to match current reality, drift trends toward zero, and a false
negative becomes indistinguishable from a healthy result.

Iceberg's answer is time travel: every write leaves an immutable snapshot. This
cell resolves the ID of each table's current snapshot, and the next cell reads
through those IDs.

Two implementation details explain the code:

**It reads `$history`, not `$snapshots`.** `$snapshots` lists every snapshot ever
written, including orphans left behind by a rollback. `$history` with
`is_current_ancestor` answers the question we actually care about -- which
snapshot is the current state's ancestor. Both are Athena metadata tables, so no
extra library is needed.

**It refuses to continue unpinned.** If either ID cannot be resolved,
`REQUIRE_PINNED_BASELINE = True` raises instead of quietly degrading. Should it
fire, check that Lab 2 has written both tables, then re-run. Setting the flag to
`False` proceeds anyway, but the model package will be registered
`PendingManualApproval` so a degraded baseline cannot become Lab 5's fallback.

**What you should see**: two 19-digit snapshot IDs, one per table.

In [ ]:
# Freeze the Iceberg baseline for drift detection.
#
# Drift detection asks whether live traffic still resembles the data the model
# was trained on -- which means it needs that data AS IT WAS AT TRAINING TIME.
# `training_data` / `evaluation_data` are live Iceberg tables that Lab 2
# overwrite()s, so "the training distribution" moves every time Lab 2 re-runs.
# Iceberg's answer is time travel: every write produces an immutable snapshot.
#
# We resolve each table's current snapshot here and then read the training CSVs
# through it, so the pinned snapshot is by construction the data this model was
# trained on. Lab 2 prints these same IDs after it writes the tables, but nothing
# needs to be carried over from there -- pinning at read time is what keeps the
# guarantee true even if Lab 2 has been re-run since.

# An unpinned baseline is the failure this whole section exists to prevent: it
# moves under the drift monitor and drives drift toward zero, a false negative
# indistinguishable from a healthy run. So a snapshot that cannot be resolved
# stops the lab rather than degrading quietly. Set this to False to override.
REQUIRE_PINNED_BASELINE = True


def _current_snapshot_id(table):
    """Current-lineage Iceberg snapshot ID for `table`, or '' if unreadable.

    Reads `$history` rather than `$snapshots`. `$snapshots` holds every snapshot
    ever written, including ones that are no longer ancestors of the current
    state (after a rollback), so "newest committed" and "current" are different
    questions -- `is_current_ancestor` asks the second one. Athena exposes both
    as metadata tables, so this needs no PyIceberg dependency.
    """
    q = (f'SELECT CAST(snapshot_id AS VARCHAR) AS sid '
         f'FROM "{ATHENA_DATABASE}"."{table}$history" '
         f'WHERE is_current_ancestor ORDER BY made_current_at DESC LIMIT 1')
    try:
        d = wr.athena.read_sql_query(
            q, database=ATHENA_DATABASE, s3_output=athena_output, ctas_approach=False
        )
    except Exception as e:
        print(f'  {table}: could not read $history ({type(e).__name__}: {e})')
        return ''
    if d.empty:
        print(f'  {table}: no snapshots yet -- has Lab 2 written this table?')
        return ''
    return str(d['sid'].iloc[0])


training_snapshot_id = _current_snapshot_id(TRAINING_TABLE)
evaluation_snapshot_id = _current_snapshot_id(EVALUATION_TABLE)

print('Frozen baseline for Lab 5:')
print(f'  {TRAINING_TABLE:16s} snapshot {training_snapshot_id or "(NONE)"}')
print(f'  {EVALUATION_TABLE:16s} snapshot {evaluation_snapshot_id or "(NONE)"}')

if not (training_snapshot_id and evaluation_snapshot_id):
    _msg = (
        'Could not pin an Iceberg snapshot for both tables, so drift would be '
        'measured against the LIVE table. Check that Lab 2 has written both '
        'tables and that this role can query their $history metadata, then '
        're-run. To proceed anyway, set REQUIRE_PINNED_BASELINE = False above; '
        'the model package will then be registered PendingManualApproval so it '
        "cannot become Lab 5's fallback baseline."
    )
    if REQUIRE_PINNED_BASELINE:
        raise RuntimeError(_msg)
    print(f'\nWARNING: {_msg}')

### Step 3.3: Materialize CSVs through the pinned snapshot

The XGBoost container reads CSV from S3, not Iceberg, so the tables have to be
materialized. The `FOR VERSION AS OF <snapshot_id>` clause is the time travel
from the previous step in action: the CSV is built from the pinned snapshot,
which makes the recorded snapshot ID a true description of what the model
trained on rather than a hopeful annotation.

Two format details are the XGBoost container's requirements, not ours:

- **Target first.** Column 0 is the label (`subscribed` cast to integer), features follow in `FEATURES` order.
- **No header row.** Which is exactly why column order is a contract -- the model will only ever know these columns by position.

**What you should see**: the `FROM ... FOR VERSION AS OF ...` clause echoed for
each table, plus row counts (roughly 33k train / 8k test).

In [ ]:
# Materialize the training CSVs from the PINNED snapshots.
#
# `FOR VERSION AS OF <snapshot_id>` is Iceberg time travel: it reads the table
# exactly as it existed at that snapshot, regardless of what has been written
# since. Because the CSVs the training job consumes come from the same pinned
# snapshot recorded in baseline.json, "the data this model was trained on" is
# reproducible rather than a moving target.
def _iceberg_to_training_csv(table, dest_key, snapshot_id=''):
    """Query an Iceberg table and write a target-first, header-less CSV to S3."""
    src = f'{table} FOR VERSION AS OF {snapshot_id}' if snapshot_id else table
    q = 'SELECT CAST(subscribed AS integer) AS y, ' + ', '.join(FEATURES) + f' FROM {src}'
    d = wr.athena.read_sql_query(q, database=ATHENA_DATABASE, s3_output=athena_output, ctas_approach=False)
    buf = io.StringIO()
    d.to_csv(buf, index=False, header=False)
    boto3.client('s3', region_name=region).put_object(Bucket=bucket, Key=dest_key, Body=buf.getvalue())
    print(f'  FROM {src}')
    print(f'    -> s3://{bucket}/{dest_key} ({d.shape[0]:,} rows)')
    return d


train_key = f'{prefix}/data/train/train.csv'
test_key = f'{prefix}/data/test/test.csv'
_iceberg_to_training_csv(TRAINING_TABLE, train_key, training_snapshot_id)
_eval_df = _iceberg_to_training_csv(EVALUATION_TABLE, test_key, evaluation_snapshot_id)

train_s3 = f's3://{bucket}/{train_key}'
test_s3 = f's3://{bucket}/{test_key}'
print(f'\nTrain S3: {train_s3}\nTest S3: {test_s3}')

### Step 3.4: Keep a local hold-out

The evaluation set stays in memory, split into `X_test` (features) and `y_test`
(labels). It gets used twice more: Section 5 scores the trained booster on it to
compute the baseline metrics, and Section 6 sends a few rows to the live
endpoint.

Because it comes from the pinned evaluation snapshot, the baseline metrics
describe exactly the rows `evaluation_snapshot_id` names.

In [ ]:
# Local hold-out set for endpoint testing (Section 6).
# _eval_df is target-first (col 0 = y), remaining cols are FEATURES in order.
y_test = _eval_df.iloc[:, 0]
X_test = _eval_df.iloc[:, 1:]
print(f'Loaded eval set from Iceberg - X_test: {X_test.shape}, y_test: {y_test.shape}')

---

## Section 4: Model Training with ModelTrainer

### What You'll Learn

In this section, you'll:
1. Create a training script for XGBoost
2. Configure the ModelTrainer with hyperparameters
3. Launch a SageMaker training job
4. Monitor training progress and view results

### Key Concepts

**ModelTrainer (SageMaker SDK v3)**: Simplified training API that:
- Manages training infrastructure automatically
- Handles script packaging and dependencies
- Integrates with MLflow for tracking
- Provides intelligent defaults
- Supports distributed training

**SageMaker Training Jobs**: Managed training infrastructure that:
- Provisions compute instances automatically
- Downloads data from S3
- Runs your training script
- Uploads model artifacts to S3
- Terminates instances when done thus cost-efficient


### Steps and Instructions

1. **Create training script**: We'll write a Python script that trains XGBoost
2. **Configure ModelTrainer**: Set up compute resources and hyperparameters
3. **Start training**: Launch the job (takes ~5 minutes)
4. **Monitor progress**: Watch logs in SageMaker AI Studio training job in real-time
5. **View results**: Check MLflow for metrics and artifacts

Run the cells below to train your model!


### Step 4.1: The training script

The next two cells write files to a local `scripts/` directory. Nothing trains
yet -- `ModelTrainer` uploads this directory to S3 and the training container
runs it there.

`train.py` is written as a string and saved to disk. Inside the container it:

1. Reads `/opt/ml/input/data/train/train.csv` and `.../test/test.csv` with `header=None`, splitting column 0 off as the label -- the other half of Step 3.3's format contract.
2. Points MLflow at the tracking URI passed in as an environment variable, so runs land in the managed app rather than inside the container.
3. Enables `mlflow.xgboost.autolog(log_models=False)`, which captures parameters and metrics automatically but leaves the model to an explicit call.
4. Tags the run with its SageMaker training job name, so Section 5 can find *this* run rather than "the most recent one in the experiment".
5. Trains, evaluates (accuracy, precision, recall, F1, AUC), and logs the metrics.
6. Logs the model with `mlflow.xgboost.log_model(...)`, which creates an MLflow **logged model** -- an entity with its own `model_id` and artifact location. That artifact location is what Section 5 turns into a deployable Model Package, and what the endpoint serves directly from.

**Why `log_models=False` and then log explicitly?** Two reasons, both about
control. Autologging would log the model as a side effect *and*, if given
`registered_model_name`, register it -- and registration is exactly what has to
happen last, after Section 5 has attached the serving code and the inference
specification to the model. A model registered before that is a Model Package
with nothing to serve. Logging explicitly also returns a `ModelInfo` with the
`model_id`, which is the handle everything in Section 5 hangs off.

`model_format='json'` makes the on-disk file name predictable
(`model.json`), which the inference handler's `model_fn` relies on.

`requirements.txt` is installed into the training container before your script
runs. The container gets its own dependency set and does not inherit the
notebook's, which is why the MLflow version there is pinned separately from the
one installed in Step 1.1.


In [ ]:
# Create training script directory
os.makedirs('scripts', exist_ok=True)

training_script = '''import argparse
import os
import json
import logging
import sys
import xgboost as xgb
import pandas as pd
import mlflow
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.addHandler(logging.StreamHandler(sys.stdout))

# The MLflow registered-model name. Registration itself happens in the notebook
# (Section 5), AFTER the serving code and inference specification are attached to
# the logged model -- registering here would produce a Model Package with no
# inference specification, which cannot be deployed.
MLFLOW_MODEL_NAME = 'bank-prediction-XGBoostModel'


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--max_depth', type=int, default=5)
    parser.add_argument('--eta', type=float, default=0.2)
    parser.add_argument('--gamma', type=int, default=4)
    parser.add_argument('--min_child_weight', type=int, default=6)
    parser.add_argument('--subsample', type=float, default=0.8)
    parser.add_argument('--num_round', type=int, default=100)
    return parser.parse_known_args()


def training_job_name():
    \'\'\'The SageMaker training job this script is running in, or \'local\'.\'\'\'
    try:
        return json.loads(os.environ.get('SM_TRAINING_ENV', '{}'))['job_name']
    except Exception:
        return os.environ.get('TRAINING_JOB_NAME', 'local')

if __name__ == '__main__':
    args, _ = parse_args()
    
    # Load data
    train_data = pd.read_csv('/opt/ml/input/data/train/train.csv', header=None)
    test_data = pd.read_csv('/opt/ml/input/data/test/test.csv', header=None)
    
    X_train, y_train = train_data.iloc[:, 1:], train_data.iloc[:, 0]
    X_test, y_test = test_data.iloc[:, 1:], test_data.iloc[:, 0]
    
    # DMatrix from numpy, not from the DataFrame: a DataFrame would stamp its
    # column labels (0..19, from the header-less CSV) onto the booster as feature
    # names, and the serving path builds its DMatrix positionally. xgboost then
    # rejects the prediction with \'training data did not have the following
    # fields\'. Positional in, positional out.
    dtrain = xgb.DMatrix(X_train.values, label=y_train)
    dtest = xgb.DMatrix(X_test.values, label=y_test)
    # Set MLFlow specifics
    mlflow_app_arn = os.environ.get('MLFLOW_TRACKING_URI', None)
    mlflow_experiment_name = os.environ.get('MLFLOW_EXP', None)
    logger.info(f'MLFLOW URI: {mlflow_app_arn}')
    logger.info(f'MLFLOW Exp: {mlflow_experiment_name}')
    
    # MLflow setup
    try:
        mlflow.set_tracking_uri(mlflow_app_arn)
        logger.info(f'Tracking URI set successfully. Active URI: {mlflow.get_tracking_uri()}')
    except Exception as e:
        logger.error(f'Failed to set tracking URI: {e}', exc_info=True)
    
    try:
        mlflow.set_experiment(mlflow_experiment_name)
        exp = mlflow.get_experiment_by_name(mlflow_experiment_name)
        logger.info(f'Experiment set. ID: {exp.experiment_id}, Name: {exp.name}, Lifecycle: {exp.lifecycle_stage}')
    except Exception as e:
        logger.error(f'Failed to set experiment: {e}', exc_info=True)
    
    # Autolog parameters and metrics, but NOT the model: the model is logged
    # explicitly below so this script gets back its model_id, and so nothing is
    # registered before Section 5 attaches the inference specification.
    mlflow.xgboost.autolog(
        log_input_examples=True,
        log_model_signatures=True,
        log_models=False,
        log_datasets=True,
        extra_tags={"team": "data-science"},
    )
    logger.info('Autologging enabled (log_models=False)')
    
    # MLflow tracking
    try:
        run = mlflow.start_run()
        logger.info(f'MLflow run started. Run ID: {run.info.run_id}, Experiment ID: {run.info.experiment_id}, Status: {run.info.status}')
    except Exception as e:
        logger.error(f'Failed to start MLflow run: {e}', exc_info=True)
        raise
    with run:
        # The notebook finds THIS run by this tag. Without it, Section 5 would
        # have to guess at "the latest run in the experiment" -- which is the
        # wrong answer whenever two people share the experiment.
        job_name = training_job_name()
        mlflow.set_tag('sagemaker.training_job_name', job_name)
        logger.info(f'Tagged run with training job name: {job_name}')

        params = {
            'max_depth': args.max_depth,
            'eta': args.eta,
            'gamma': args.gamma,
            'min_child_weight': args.min_child_weight,
            'subsample': args.subsample,
            'objective': 'binary:logistic',
            'eval_metric': 'auc'
        }
        
        mlflow.log_params(params)
        
        # Train
        model = xgb.train(params, dtrain, args.num_round, evals=[(dtest, 'test')])
        
        # Evaluate
        y_pred_proba = model.predict(dtest)
        y_pred = (y_pred_proba > 0.5).astype(int)
        
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'f1': f1_score(y_test, y_pred),
            'auc': roc_auc_score(y_test, y_pred_proba)
        }
        
        mlflow.log_metrics(metrics)
        logger.info(f'Metrics logged: {metrics}')
        print(f'Metrics: {metrics}')
        
        # Log the model explicitly. This creates the MLflow LOGGED MODEL whose
        # model_id Section 5 attaches inference.py and the inference
        # specification to, and whose artifact location the endpoint serves from.
        # model_format='json' fixes the file name to model.json, which the
        # inference handler's model_fn looks for.
        model_info = mlflow.xgboost.log_model(
            model,
            name=MLFLOW_MODEL_NAME,
            model_format='json',
        )
        logger.info(f'Logged model {model_info.model_id} -> {model_info.model_uri}')
        print(f'MLflow logged model id: {model_info.model_id}')
    
    logger.info(f'MLflow run completed. Active run: {mlflow.active_run()}')
'''

with open('scripts/train.py', 'w') as f:
    f.write(training_script)
    
print('\u2713 Training script created')


In [ ]:
# Create requirements.txt for the training script
requirements_content = """mlflow>=3,<4
sagemaker-mlflow>=0.5.0,<1
scikit-learn
xgboost>=3,<4
pandas>=2,<3
numpy>=2,<3
"""

with open('scripts/requirements.txt', 'w') as f:
    f.write(requirements_content)

print('✓ requirements.txt created')


### Step 4.2: Which container runs the script

`image_uris.retrieve` resolves the AWS-managed XGBoost 1.7-1 image for your
region. This is the *training* variant (`image_scope='training'`); Section 5
retrieves the inference variant of the same version.

Keeping both at 1.7-1 matters: the booster is serialized by the training
container and deserialized by the serving container, so a version skew between
them is a load failure at deploy time.

In [ ]:
# Get XGBoost container image
xgboost_image = image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='3.0-5',
    image_scope='training',
    instance_type="ml.m5.xlarge",
)
print(f'XGBoost image: {xgboost_image}')

### Step 4.3: Describe the job

`ModelTrainer` is declarative -- this cell describes the job without starting
anything. Four pieces:

- **`SourceCode`** -- the `scripts/` directory to upload and `train.py` as the entry point
- **`Compute`** -- one `ml.m5.xlarge`, 30 GB of volume, provisioned for the job and torn down after
- **`hyperparameters`** -- passed to the script as command-line flags, which is why `train.py` parses them with `argparse`
- **`environment`** -- how `mlflow_app_arn` and the experiment name reach the container, where `train.py` reads them with `os.environ.get`

`base_job_name='bank-marketing-xgboost'` gets a timestamp appended, so each run
produces a uniquely named job you can find in Studio.

In [ ]:
# Configure ModelTrainer with v3 API
source_code = SourceCode(
    source_dir='scripts',
    entry_script='train.py',
    requirements="requirements.txt"
)

compute = Compute(
    instance_type='ml.m5.xlarge',
    instance_count=1,
    volume_size_in_gb=30
)

hyperparameters = {
    'max_depth': 5,
    'eta': 0.2,
    'gamma': 4,
    'min_child_weight': 6,
    'subsample': 0.8,
    'num_round': 100
}

model_trainer = ModelTrainer(
    sagemaker_session=sagemaker_session,
    training_image=xgboost_image,
    source_code=source_code,
    compute=compute,
    hyperparameters=hyperparameters,
    base_job_name='bank-marketing-xgboost',
    environment={'MLFLOW_TRACKING_URI': mlflow_app_arn,
                 'MLFLOW_EXP': mlflow_experiment_name
                }
)

print('ModelTrainer configured')

### Step 4.4: Launch and wait

`InputData` channels become directories inside the container: the `train`
channel appears at `/opt/ml/input/data/train/`, which is the path `train.py`
reads. Channel names and script paths have to agree.

`wait=True` blocks and streams the container's logs into the cell output, so you
watch the AUC per boosting round as it happens. Expect about 5 minutes, most of
it spent provisioning the instance rather than training.

While it runs, open **Training -> Training jobs** in Studio and find the
`bank-marketing-xgboost-*` job.

**What you should see**: log lines ending with a metrics dictionary, then
`Training completed: bank-marketing-xgboost-<timestamp>`. Once the job reaches
`Completed`, the model tarball is in S3 and Section 5 can pick it up.

In [ ]:
# Start training
input_data_train = InputData(channel_name='train', data_source=train_s3)
input_data_test = InputData(channel_name='test', data_source=test_s3)

# Go the the sagemaker Studio training to find the training job in-progress with name "bank-marketing-xgboost*"
training_job = model_trainer.train(
    input_data_config=[input_data_train, input_data_test],
    wait=True
)

# ModelTrainer exposes the completed job as `_latest_training_job` (train() may
# also return it, depending on the SDK version) - handle both.
training_job = training_job or model_trainer._latest_training_job
print(f'Training completed: {training_job.training_job_name}')

### Understanding the Training Output

Now let's explore what was automatically logged to MLflow during training!

**Navigate to MLflow UI**
1. In SageMaker Studio, click on **MLflow** in the left sidebar
2. Find your MLflow app (DefaultMLFlowApp)
3. Click to open the MLflow UI

**View the Experiment**
1. In MLflow UI, click on **Experiments** tab
2. Find the experiment: **bank-marketing-prediction**
3. You'll see your training run listed with:
   - Run name and ID
   - Start time and duration
   - Status (Finished)
   - Metrics preview

**Explore the Run Details**

Click on the run to see comprehensive details:

- **Parameters Tab**: all hyperparameters used
- **Metrics Tab**: training metrics over time
- **Artifacts Tab**: the logged model -- `model.json`, `MLmodel`, `conda.yaml`, `requirements.txt`
- **Models Tab**: the **logged model** produced by `mlflow.xgboost.log_model()`, with its own `model_id` (`m-...`)

**Two registries, and what is in each right now**

MLflow has a model registry; SageMaker AI has one too. At this point in the
notebook **both are empty for this run** -- training logged a model, but
registered nothing anywhere. That is deliberate.

Section 5 registers once, in MLflow. Because the app runs with
`AutoModelRegistrationEnabled` (checked in Step 2.1), SageMaker mirrors that
registration into its own Model Registry as a **Model Package**, carrying the
serving container, the model location, and the environment that Section 5
attaches first. One registration action, both registries, and the result is
deployable.

The ordering is the whole point: register before attaching the inference
specification and you get a Model Package that cannot be deployed.

### Next Steps

1. **Compare Runs**: Train with different hyperparameters and compare in MLflow
2. **Register**: Make the logged model deployable, then register it (Section 5)
3. **Deploy Model**: Deploy the synced Model Package to an endpoint

Now let's make the model deployable, register it with a frozen drift baseline, and deploy it!


---

## Section 5: Register and Deploy the Model from MLflow

### What You'll Learn

In this section, you'll:

1. Resolve the MLflow **logged model** your training job produced
2. Author a custom **script-mode** inference handler (`inference.py`) and log it *into the model's own artifact store*
3. Validate that handler in this kernel, and score the pinned evaluation snapshot with it
4. Log an **inference specification** on the logged model -- the container, the model location, the environment
5. **Register once, in MLflow**, and watch SageMaker auto-sync a deployable **Model Package**
6. Attach the frozen **`baseline.json`** to that package and **approve** it
7. Wire the endpoint to the **inference-capture** plane (SQS -> Lambda -> Iceberg) and deploy

### Key Concepts

**The MLflow artifact store is the single source of truth.** There is no
repacking and no copy of the model bytes. The Model Package points a
`ModelDataSource` at the logged model's artifact location with
`S3DataType: S3Prefix` and `CompressionType: None`, and the endpoint downloads
that whole directory to `/opt/ml/model` at startup:

```
MLflow artifact store ◀────────── serves directly from ──────────┐
        │  log_inference_specification()                         │
        ▼                                                        │
MLflow ──register──▶ auto-sync ──▶ Model Package (deployable) ──▶ Endpoint
```

**A Model Package is only deployable if it carries an inference
specification** -- the serving image, the model data location, and the
environment variables the serving stack needs. So before registering you log
that specification onto the MLflow model with the `sagemaker-mlflow` plugin, and
the auto-sync copies it onto the package.

<div class="alert alert-warning">
<b>Order matters:</b> the specification must be logged <b>before</b>
<code>mlflow.register_model()</code>. The auto-sync copies it onto the Model
Package at registration time -- log it afterwards and the package already exists
without it.
</div>

**Custom inference handler (script mode)**: the stock XGBoost container serves
raw predictions and captures nothing. A file logged at `code/inference.py` in the
artifact store arrives at `/opt/ml/model/code/inference.py` on the endpoint, and
`SAGEMAKER_PROGRAM` in the specification's `Environment` tells the container to
run it. The handler returns structured predictions AND publishes one message per
prediction to SQS, which a logger Lambda drains into the Athena/Iceberg
`inference_responses` table Lab 5 reads.

**`baseline.json`**: a small JSON artifact holding the Iceberg snapshot IDs,
table names, feature schema and evaluation metrics that define this model
version's reference point. It is attached to the Model Package under
`CustomerMetadataProperties['baseline_s3_uri']`, which is what lets Lab 5 find it
later -- the package is the durable record, so the baseline travels with the
model rather than living in someone's notes.

> `ModelMetrics` is the field you would normally use for that, but it can only be
> set at `CreateModelPackage` time and here SageMaker creates the package for us.
> `UpdateModelPackage` accepts `CustomerMetadataProperties`, so that is where the
> pointer goes -- and it shows up on the version's **Custom metadata** in Studio.

**Register before deploy**: Lab 5 identifies a model's baseline by asking the
*running* endpoint which Model Package it serves. That only works if the
SageMaker Model was created **from** a package -- a model built directly from
`Image` + `ModelDataUrl` carries no `ModelPackageName` and the lookup dead-ends.

**SageMaker Endpoints**: real-time, always-on HTTPS inference with auto-scaling,
CloudWatch metrics, and VPC/IAM security.

> **Important**: Endpoints run continuously and incur charges. Always delete
> them when not in use (see the cleanup cell at the end).

### Instructions

1. **Resolve the logged model** from the training job's MLflow run
2. **Get the serving image**
3. **Write `inference.py`** (the custom handler with SQS capture)
4. **Log `code/inference.py` + `feature_names.json`** into the model's artifact store
5. **Validate the handler locally** and score the pinned evaluation snapshot
6. **Resolve serving/capture configuration** while every value is known locally
7. **Log the inference specification**, including the complete container environment
8. **Register in MLflow** and wait for the synced Model Package
9. **Write `baseline.json`**, attach it to the package, and approve
10. **Deploy** the approved package to an endpoint (5-7 minutes)

Run the cells below to register and deploy your model!


### Step 5.1: Locate the logged model in MLflow

Everything in this section hangs off one identifier: the `model_id` of the
MLflow **logged model** `train.py` produced. MLflow 3 makes that a first-class
entity -- it has its own artifact location, its own tags, and it is what
`mlflow.register_model()` registers.

The cell finds it in two hops, both exact rather than "most recent":

1. the run tagged `sagemaker.training_job_name = <this job>` (Step 4.1 set that tag)
2. the logged model whose `source_run_id` is that run

Why the fuss: the workshop experiment is shared between `userA` and `userB`, and
`search_runs(order_by=start_time DESC)` would happily hand you your colleague's
run. Resolving through the training job name you just launched cannot do that.

**What you should see**: a run id, a logged model id starting `m-`, and an
`s3://...-mlflow-.../models/m-.../artifacts` location. That S3 prefix is what the
endpoint will serve from -- no repacking, no copies.


In [ ]:
# Resolve the MLflow logged model produced by the training job above.
#
# The endpoint serves straight out of this artifact location, so from here on the
# MLflow artifact store is the single source of truth for the model bytes.
training_job_name = model_trainer._latest_training_job.training_job_name
print(f'Training job: {training_job_name}')

# The MLflow registered-model name, matching MLFLOW_MODEL_NAME in train.py. The
# SageMaker model package group the auto-sync creates is this name plus a
# generated suffix, resolved in Step 5.7.
mlflow_model_name = 'bank-prediction-XGBoostModel'

mlflow_client = MlflowClient()
experiment_id = mlflow.get_experiment_by_name(mlflow_experiment_name).experiment_id

# Find THIS job's run by the tag train.py set, not by recency: the experiment is
# shared across user profiles, so "the latest run" is somebody else's model
# often enough to matter.
_runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=f"tags.`sagemaker.training_job_name` = '{training_job_name}'",
    max_results=1,
)
assert len(_runs), (
    f'No MLflow run tagged with training job {training_job_name}. Did Step 4.4 '
    f'finish, and is train.py the version from Step 4.1?')
mlflow_run_id = _runs['run_id'].iloc[0]

# The MLflow 3 logged model carries the model_id and artifact_location that the
# inference specification points at.
logged_model = next(
    m for m in mlflow.search_logged_models(experiment_ids=[experiment_id], output_format='list')
    if m.source_run_id == mlflow_run_id
)

print(f'  MLflow run        : {mlflow_run_id}')
print(f'  Logged model id   : {logged_model.model_id}')
print(f'  Artifact location : {logged_model.artifact_location}')


### Step 5.2: The serving image

The same XGBoost `3.0-5`, this time `image_scope='inference'`. The serving image
ships the model server that will load our handler; the training image does not.

Keeping both at the same version matters: the booster is serialized by the
training container and deserialized by the serving container, so a version skew
between them is a load failure at deploy time.


In [ ]:
# Get inference image URI
inference_image = image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='3.0-5',
    image_scope='inference'
)
print(f'Inference image: {inference_image}')

### Step 5.3: The inference handler

The longest cell in the notebook, and the most important one to understand. It
writes `scripts/inference.py`. Nothing executes here -- this code runs *inside
the endpoint*, once deployed.

Why a custom handler at all? The stock XGBoost container answers with a bare
probability and records nothing. Lab 5 needs every prediction persisted, with
enough context to join it back to the model that made it. So the handler does
two jobs: serve, and capture.

SageMaker's script mode calls four functions in order, and the handler
implements each:

| Function | Responsibility |
|---|---|
| `model_fn` | Load the booster and `feature_names.json` once at container start |
| `input_fn` | Parse the request -- `application/json` with named features, or `text/csv` positional |
| `predict_fn` | Order features, predict, and publish one capture message per prediction |
| `output_fn` | Serialize `{"predictions": [...], "probabilities": {"yes": [...], "no": [...]}}` |

Four details are load-bearing:

**`model_fn` receives the MLflow model directory.** The specification's
`S3Prefix` `ModelDataSource` downloads the *whole* logged-model directory to
`/opt/ml/model`, so `model.json`, `MLmodel` and `conda.yaml` all land there. The
handler loads the booster with plain `xgboost` rather than
`mlflow.xgboost.load_model()` on purpose -- MLflow is not installed in the stock
XGBoost serving container, and adding it would mean a pip install on every cold
start.

**`predict_fn` builds its `DMatrix` from a numpy array, not a DataFrame.** The
booster trained on header-less CSV, so it knows features as `f0..f19`. Handing
it a named DataFrame raises a feature-name mismatch. Ordering comes from
`feature_names.json`; the names are then only used to label captured rows.

**Capture is fire-and-forget.** The SQS send is wrapped so that a failure prints
and moves on. Monitoring must never take down serving.

**Configuration arrives through the environment.** `SQS_QUEUE_URL`,
`MODEL_VERSION`, `MLFLOW_RUN_ID`, `FEATURE_NAMES` and the confidence thresholds
are all read with `os.getenv`. The handler discovers nothing on its own -- Step
5.6 resolves those values and Step 5.7 records them once in the inference
specification. If the queue URL is empty, capture disables itself and predictions
continue.

The code avoids type hints and modern syntax on purpose: the XGBoost container
ships an older Python than this notebook.


In [ ]:
# Custom inference handler (script mode) - written inline, same pattern as
# train.py above. The stock XGBoost container serves raw predictions and
# captures nothing; this handler adds inference capture: each prediction is
# published to the SQS -> Lambda -> Iceberg plane provisioned by
# templates/4-inference-capture.yaml (endpoint handler --1 msg/prediction-->
# SQS --> logger Lambda --> Athena INSERT into
# bank_marketing.inference_responses).
#
# Step 5.4 logs this file into the MLflow model's artifact store at
# code/inference.py, where the endpoint finds it as /opt/ml/model/code/inference.py
# and runs it via SAGEMAKER_PROGRAM. A copy is kept in scripts/ so you can read it
# alongside train.py.
os.makedirs('scripts', exist_ok=True)

inference_script = r"""
import json
import os
import time
import uuid
import logging
from datetime import datetime

import pandas as pd
import numpy as np
import xgboost as xgb
import boto3

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Configuration from container environment ---
# INTERIM: the Lab 3A deploy cell injects these as container env vars. The
# planned end-state is to manage them centrally as environment variables;
# for now the notebook resolves and passes them at deploy time.
ENABLE_INFERENCE_CAPTURE = os.getenv('ENABLE_INFERENCE_CAPTURE', 'true').lower() == 'true'
ENDPOINT_NAME = os.getenv('ENDPOINT_NAME', 'unknown')
MODEL_VERSION = os.getenv('MODEL_VERSION', 'unknown')
MLFLOW_RUN_ID = os.getenv('MLFLOW_RUN_ID', 'unknown')
SQS_QUEUE_URL = os.getenv('SQS_QUEUE_URL', '')
HIGH_CONFIDENCE_THRESHOLD = float(os.getenv('HIGH_CONFIDENCE_THRESHOLD', '0.9'))
LOW_CONFIDENCE_LOWER = float(os.getenv('LOW_CONFIDENCE_LOWER', '0.4'))
LOW_CONFIDENCE_UPPER = float(os.getenv('LOW_CONFIDENCE_UPPER', '0.6'))

sqs_client = None


def _region_from_queue_url(url):
    # URL shape: https://sqs.<region>.amazonaws.com/<account>/<queue>
    # Untyped on purpose: the XGBoost container ships Python 3.9.
    try:
        host = url.split('//', 1)[1].split('/', 1)[0]
        parts = host.split('.')
        if len(parts) >= 3 and parts[0] == 'sqs':
            return parts[1]
    except Exception:
        pass
    return None


def get_sqs_client():
    global sqs_client
    if sqs_client is None:
        region = (_region_from_queue_url(SQS_QUEUE_URL)
                  or os.getenv('SAGEMAKER_REGION')
                  or os.getenv('AWS_REGION'))
        if not region:
            raise RuntimeError('Cannot determine AWS region for SQS client.')
        sqs_client = boto3.client('sqs', region_name=region)
    return sqs_client


def get_prediction_bucket(p):
    if p < 0.2:
        return 'very_low'
    if p < 0.4:
        return 'low'
    if p < 0.6:
        return 'medium'
    if p < 0.8:
        return 'high'
    return 'very_high'


def model_fn(model_dir):
    # model_dir is /opt/ml/model, into which the S3Prefix ModelDataSource
    # downloaded the WHOLE MLflow logged-model directory: the booster written by
    # mlflow.xgboost.log_model(model_format='json') plus MLmodel, conda.yaml and
    # the code/ directory this file came from.
    #
    # The booster is loaded with plain xgboost rather than
    # mlflow.xgboost.load_model(): mlflow is not installed in the stock XGBoost
    # serving container, and installing it would add a pip step to every cold
    # start. Booster.load_model() sniffs the format, so json / ubj / legacy
    # binary all work. Several names are tried so an artifact produced by an
    # older notebook run (or a repacked tarball) still loads.
    start = time.time()
    model_path = None
    for fn in ['model.json', 'model.ubj', 'model.xgb', 'model.pkl',
               'xgboost-model.json', 'xgboost-model',
               os.path.join('data', 'model.json'),
               os.path.join('data', 'model.xgb')]:
        p = os.path.join(model_dir, fn)
        if os.path.exists(p):
            model_path = p
            break
    if model_path is None:
        raise FileNotFoundError(
            'No model file found in ' + model_dir + ' (contents: '
            + str(sorted(os.listdir(model_dir))) + ')')
    if model_path.endswith('.pkl'):
        import pickle
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
    else:
        model = xgb.Booster()
        model.load_model(model_path)
    # feature_names.json is logged at the artifact root (Step 5.4), so it lands
    # next to the model; code/ is checked too for older layouts.
    feature_names = None
    for meta in [os.path.join(model_dir, 'feature_names.json'),
                 os.path.join(model_dir, 'code', 'feature_names.json')]:
        if os.path.exists(meta):
            with open(meta) as f:
                feature_names = json.load(f)['feature_names']
            break
    load_ms = (time.time() - start) * 1000
    logger.info('Model loaded in %.1fms; %d features', load_ms, len(feature_names or []))
    return {'model': model, 'feature_names': feature_names, 'model_load_time_ms': load_ms}


def input_fn(request_body, content_type='application/json'):
    if isinstance(request_body, (bytes, bytearray)):
        request_body = request_body.decode('utf-8')
    if content_type == 'application/json':
        data = json.loads(request_body)
        if isinstance(data, dict):
            df = pd.DataFrame([data])
        elif isinstance(data, list):
            df = pd.DataFrame(data)
        else:
            raise ValueError('Unsupported JSON payload: ' + str(type(data)))
    elif content_type == 'text/csv':
        from io import StringIO
        df = pd.read_csv(StringIO(request_body), header=None)
        # Positional CSV -> name columns from FEATURE_NAMES so capture records
        # carry named features (matching the training_data baseline).
        names = os.getenv('FEATURE_NAMES')
        if names:
            cols = [c.strip() for c in names.split(',')]
            if len(cols) == df.shape[1]:
                df.columns = cols
    else:
        raise ValueError('Unsupported content type: ' + str(content_type))
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def predict_fn(input_data, model_dict):
    start = time.time()
    model = model_dict['model']
    feature_names = model_dict['feature_names']
    # Fall back to incoming column order if the artifact had no feature_names.
    if not feature_names:
        feature_names = list(input_data.columns)
    for f in feature_names:
        if f not in input_data.columns:
            input_data[f] = 0.0
    ordered = input_data[feature_names]
    preprocessing_ms = (time.time() - start) * 1000

    # DMatrix from a numpy array: the booster is trained on a header-less
    # positional CSV, so it carries no feature names and name-based validation
    # would mismatch. A booster that DOES carry names (trained from a DataFrame,
    # e.g. by an older version of train.py) is fed the same positional values
    # under its own names, which keeps this handler working either way.
    _booster_features = getattr(model, 'feature_names', None)
    if _booster_features and len(_booster_features) == ordered.shape[1]:
        dmatrix = xgb.DMatrix(ordered.values, feature_names=list(_booster_features))
    else:
        dmatrix = xgb.DMatrix(ordered.values)
    probabilities = model.predict(dmatrix)
    predictions = (probabilities > 0.5).astype(int)

    results = {
        'predictions': predictions.tolist(),
        'probabilities': {
            'yes': probabilities.tolist(),
            'no': (1 - probabilities).tolist(),
        },
    }
    latency_ms = (time.time() - start) * 1000

    # Fire-and-forget capture: a send failure must never fail the prediction.
    if ENABLE_INFERENCE_CAPTURE and SQS_QUEUE_URL:
        try:
            sqs = get_sqs_client()
            for i in range(len(predictions)):
                p_yes = float(probabilities[i])
                conf = max(p_yes, 1 - p_yes)
                msg = {
                    'inference_id': str(uuid.uuid4()),
                    'request_timestamp': datetime.utcnow().isoformat(),
                    'endpoint_name': ENDPOINT_NAME,
                    'model_version': MODEL_VERSION,
                    'mlflow_run_id': MLFLOW_RUN_ID,
                    'input_features': json.dumps(ordered.iloc[i].to_dict()),
                    'prediction': int(predictions[i]),
                    'probability_positive': p_yes,
                    'probability_negative': float(1 - p_yes),
                    'confidence_score': conf,
                    'ground_truth': None,
                    'ground_truth_timestamp': None,
                    'ground_truth_source': None,
                    'days_to_ground_truth': None,
                    'inference_latency_ms': latency_ms,
                    'model_load_time_ms': model_dict.get('model_load_time_ms', 0),
                    'preprocessing_time_ms': preprocessing_ms,
                    'client_id': (str(input_data.iloc[i].get('client_id', '')) or None),
                    'is_high_confidence': bool(conf > HIGH_CONFIDENCE_THRESHOLD),
                    'is_low_confidence': bool(LOW_CONFIDENCE_LOWER <= conf <= LOW_CONFIDENCE_UPPER),
                    'prediction_bucket': get_prediction_bucket(p_yes),
                    'request_id': str(uuid.uuid4()),
                    'response_time': datetime.utcnow().isoformat(),
                    'error_message': None,
                    'inference_mode': 'realtime',
                    'monitoring_run_id': None,
                }
                sqs.send_message(QueueUrl=SQS_QUEUE_URL, MessageBody=json.dumps(msg))
        except Exception as e:
            print('SQS send failed: ' + str(e))

    return results


def output_fn(prediction, accept='application/json'):
    if accept in ('application/json', '*/*', None, ''):
        return json.dumps(prediction)
    raise ValueError('Unsupported accept type: ' + str(accept))
"""

with open('scripts/inference.py', 'w') as f:
    f.write(inference_script)

# Step 5.4 logs this same string into the MLflow model's artifact store.
print('inference.py written to scripts/inference.py')
print(f'  {len(inference_script.splitlines())} lines, handlers: '
      f"{[n for n in ('model_fn', 'input_fn', 'predict_fn', 'output_fn') if 'def ' + n in inference_script]}")


### Step 5.4: Log the serving code into the model's artifact store

This is where the old "repack the tarball" step used to be. There is nothing to
repack now: `MlflowClient.log_model_artifacts()` uploads a local directory into
the logged model's own artifact store, preserving its layout. So a local
`code/inference.py` lands at `<artifact_location>/code/inference.py`, and
`feature_names.json` at the artifact root.

That matters because of how the endpoint gets its files. The inference
specification (Step 5.6) uses an `S3Prefix` `ModelDataSource` with
`CompressionType: None`, which downloads *everything* under the artifact
location to `/opt/ml/model`:

```
<artifact_location>/model.json          ->  /opt/ml/model/model.json
<artifact_location>/feature_names.json  ->  /opt/ml/model/feature_names.json
<artifact_location>/code/inference.py   ->  /opt/ml/model/code/inference.py
```

which is exactly the layout script mode expects, with `SAGEMAKER_SUBMIT_DIRECTORY`
pointing at `/opt/ml/model/code`.

Two things this buys over repacking. The model bytes are never copied, so there
is no second artifact that can drift from the registered one. And the serving
code is visible in the MLflow UI right next to the model -- you can open a model
version and read the `inference.py` that serves it.

The closing check re-lists the artifacts from the tracking server, because a
packaging mistake you do not catch here surfaces 10 minutes later as a failed
`/ping` on a deployed endpoint.


In [ ]:
# Log the serving code INTO the logged model's artifact store.
#
# log_model_artifacts() preserves the local directory layout, so code/inference.py
# lands at <artifact_location>/code/inference.py and arrives on the endpoint at
# /opt/ml/model/code/inference.py -- where SAGEMAKER_SUBMIT_DIRECTORY points.
# feature_names.json goes to the artifact root, next to the model, so model_fn
# can order incoming features to match training.
with tempfile.TemporaryDirectory() as _tmp:
    os.makedirs(os.path.join(_tmp, 'code'))
    with open(os.path.join(_tmp, 'code', 'inference.py'), 'w') as f:
        f.write(inference_script)
    with open(os.path.join(_tmp, 'feature_names.json'), 'w') as f:
        json.dump({'feature_names': FEATURES}, f)
    mlflow_client.log_model_artifacts(logged_model.model_id, _tmp)

# Verify from the tracking server, not from the local copy: a mistake here shows
# up as a failed /ping ten minutes into a deploy otherwise. Step 5.5 downloads
# this same directory and imports the script, which is the real check.
_logged = mlflow_client.get_logged_model(logged_model.model_id)
print(f'Logged into {_logged.artifact_location}:')
print('  code/inference.py')
print('  feature_names.json')


### Step 5.5: Validate the handler here, then score the pinned snapshot

Before anything is registered or deployed, check that the code the endpoint will
run actually works -- and use it to compute the baseline metrics, so the numbers
Lab 5 compares against come from the same code path that will serve traffic.

The cell reproduces the endpoint's startup faithfully: it downloads the same
artifacts the `S3Prefix` `ModelDataSource` would, imports
`code/inference.py` the way the container does, and calls `model_fn`, `input_fn`
and `predict_fn` in order. A failure here is a failure the endpoint would have
surfaced as a `/ping` timeout after several minutes of provisioning -- except it
costs nothing and takes seconds.

Capture stays off during this: `SQS_QUEUE_URL` is unset in this kernel, and the
handler disables capture when it is empty.

The six metrics become the reference Lab 5's model-drift check compares live
performance against, so they have to describe exactly the rows
`evaluation_snapshot_id` pins -- which they do, since `X_test` came from that
snapshot in Step 3.4.

**What you should see**: the downloaded artifact listing (model, `MLmodel`,
`code/`, `feature_names.json`), then ROC-AUC around 0.93-0.95. Note the class
imbalance in the printed positives count -- it is why `pr_auc` is recorded
alongside `roc_auc`.


In [ ]:
# Run the REAL serving code in this kernel, and score the pinned evaluation
# snapshot with it.
#
# The endpoint downloads the logged model's artifact directory to /opt/ml/model
# and imports code/inference.py from it. Same thing here: same files, same
# import, same four handlers -- so these baseline metrics are produced by the
# code path that will serve traffic, not by a stand-in.
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             precision_score, recall_score, roc_auc_score)

local_model_dir = mlflow.artifacts.download_artifacts(
    artifact_uri=f'models:/{logged_model.model_id}'
)
print(f'Model artifacts downloaded to: {local_model_dir}')
print(f'  contents: {sorted(os.listdir(local_model_dir))}')

_script_path = os.path.join(local_model_dir, 'code', 'inference.py')
assert os.path.exists(_script_path), f'code/inference.py missing from {local_model_dir}'

_spec = importlib.util.spec_from_file_location('user_inference_module', _script_path)
inference_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(inference_module)

# model_fn(/opt/ml/model) at worker startup -- here, the same directory locally.
local_model = inference_module.model_fn(local_model_dir)
print(f'  model_fn returned: {type(local_model["model"]).__name__} '
      f'with {len(local_model["feature_names"] or [])} feature names')

# The serving app hands input_fn the raw request body as bytes. Capture is off:
# SQS_QUEUE_URL is unset in this kernel.
_payload = X_test.to_csv(header=False, index=False).encode('utf-8')
_parsed = inference_module.input_fn(_payload, 'text/csv')
_result = inference_module.predict_fn(_parsed, local_model)

y_proba = np.asarray(_result['probabilities']['yes'], dtype=float)
y_hat = (y_proba > 0.5).astype(int)
y_true = y_test.astype(int).values

baseline_metrics = {
    'roc_auc': float(roc_auc_score(y_true, y_proba)),
    'pr_auc': float(average_precision_score(y_true, y_proba)),
    'precision': float(precision_score(y_true, y_hat, zero_division=0)),
    'recall': float(recall_score(y_true, y_hat, zero_division=0)),
    'f1_score': float(f1_score(y_true, y_hat, zero_division=0)),
    'accuracy': float(accuracy_score(y_true, y_hat)),
}

_slice = f'snapshot {evaluation_snapshot_id}' if evaluation_snapshot_id else 'LIVE table'
print(f'\nEvaluation slice: {ATHENA_DATABASE}.{EVALUATION_TABLE} ({_slice})')
print(f'  rows: {len(y_true):,}   positives: {int(y_true.sum()):,}')
for k, v in baseline_metrics.items():
    print(f'  {k:10s}: {v:.4f}')


### Step 5.6: Resolve serving and capture configuration

Every container environment value must be known **before** registration, because
the MLflow auto-sync copies the inference specification onto the Model Package at
registration time. This cell therefore resolves the capture queue and chooses the
endpoint identity now; Step 5.7 logs the complete environment once.

The capture plane (SQS queue, logger Lambda, Iceberg `inference_responses` table)
is provisioned by `templates/4-inference-capture.yaml`. Its queue is named
`{ProjectName}-inference-capture`, and `ProjectName` is derived from the Glue data
bucket rather than hardcoded.

`MODEL_VERSION` uses `logged_model.model_id`, the immutable MLflow logged-model
identity already known before registration. Lab 5 stores and groups this field as
a string label; it resolves package lineage separately through the running
endpoint's `ModelPackageName`. Using the logged-model ID avoids predicting a
registry version that has not been assigned yet.

**If the queue is not found**, capture is disabled in the specification. The
endpoint still deploys and serves; only capture is off.


In [ ]:
# Resolve the complete container environment BEFORE logging the inference
# specification. The Model Package will carry this configuration, so deployment
# does not need (and must not maintain) a second environment definition.
sqs = boto3.client('sqs', region_name=region)
account_id = boto3.client('sts', region_name=region).get_caller_identity()['Account']

# ProjectName backs the workshop resource names. Derive it from the Glue data
# bucket ("{ProjectName}-data-{account}-{region}") instead of hardcoding.
project_name = os.getenv('PROJECT_NAME', db_bucket.replace(f'-data-{account_id}-{region}', ''))
capture_queue_name = f'{project_name}-inference-capture'

try:
    sqs_queue_url = sqs.get_queue_url(QueueName=capture_queue_name)['QueueUrl']
    print(f'Resolved capture queue: {sqs_queue_url}')
except sqs.exceptions.QueueDoesNotExist:
    sqs_queue_url = ''
    print(f'WARNING: capture queue "{capture_queue_name}" not found.')
    print('  Endpoint will still deploy, but inference capture is disabled.')

# Choose deployment identity now because ENDPOINT_NAME is part of the immutable
# inference specification copied to the Model Package.
_deployment_timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
model_name = f'bank-marketing-model-{_deployment_timestamp}'
endpoint_name = f'bank-marketing-{_deployment_timestamp}'
endpoint_config_name = f'{endpoint_name}-config'
inference_instance_type = 'ml.m5.large'

# capture_config remains separately named because Section 6 uses its enable flag
# when deciding whether to verify Iceberg rows. serving_environment below is the
# only container environment definition.
capture_config = {
    'ENABLE_INFERENCE_CAPTURE': 'true' if sqs_queue_url else 'false',
    'SQS_QUEUE_URL': sqs_queue_url,
    'MODEL_VERSION': logged_model.model_id,
    'MLFLOW_RUN_ID': mlflow_run_id,
    'FEATURE_NAMES': ','.join(FEATURES),
}

serving_environment = {
    'SAGEMAKER_PROGRAM': 'inference.py',
    'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml/model/code',
    'SAGEMAKER_REGION': region,
    'AWS_REGION': region,
    'ENDPOINT_NAME': endpoint_name,
    **capture_config,
}

print('Container environment to log in the MLflow inference specification:')
for k, v in serving_environment.items():
    print(f'  {k} = {v}')


### Step 5.7: Log the inference specification

`sagemaker_mlflow.log_inference_specification()` attaches a SageMaker
`InferenceSpecification` to the logged model. The auto-sync copies it verbatim
onto the Model Package, making the package the single source of deployment
configuration.

Three parts carry weight:

- **`ModelDataSource`** points at the MLflow artifact location using
  `S3DataType: S3Prefix` and `CompressionType: None`, so there is no repacking.
- **`Environment`** contains the complete script-mode and inference-capture
  configuration resolved in Step 5.6. It is not repeated during deployment.
- **`SupportedRealtimeInferenceInstanceTypes`** includes the instance type chosen
  in Step 5.6.


In [ ]:
# Log the inference specification BEFORE registering. The auto-sync copies it
# onto the Model Package at registration time; this is the single source of the
# serving image, model location, and complete container environment.
inference_spec = {
    'Containers': [{
        'Image': inference_image,
        'ModelDataSource': {
            'S3DataSource': {
                'S3Uri': logged_model.artifact_location.rstrip('/') + '/',
                'S3DataType': 'S3Prefix',
                'CompressionType': 'None',
            }
        },
        'Environment': serving_environment,
    }],
    'SupportedContentTypes': ['application/json', 'text/csv'],
    'SupportedResponseMIMETypes': ['application/json'],
    'SupportedRealtimeInferenceInstanceTypes': ['ml.m5.large', 'ml.m5.xlarge'],
    'SupportedTransformInstanceTypes': ['ml.m5.large', 'ml.m5.xlarge'],
}

sagemaker_mlflow.log_inference_specification(
    logged_model.model_id, inference_specification=inference_spec
)
print(f'Inference specification logged on {logged_model.model_id}')
print(f'  image      : {inference_image}')
print(f'  model data : {inference_spec["Containers"][0]["ModelDataSource"]["S3DataSource"]["S3Uri"]}')
print(f'  environment keys: {sorted(serving_environment)}')


### Step 5.8: Register once, and watch the auto-sync

`mlflow.register_model()` creates a version in the **MLflow** model registry.
Because the app runs with `AutoModelRegistrationEnabled`, SageMaker then mirrors
that version into its own Model Registry as a **Model Package**, carrying the
specification from Step 5.7.

The sync is asynchronous, and it reports back in a convenient place: the
resulting Model Package ARN appears as a `sagemaker.model_package_arn` **tag on
the MLflow model version**. Polling that tag is simpler and more precise than
searching the registry.

Two things about the package SageMaker creates:

**The model package group name gets a suffix.** Registering
`bank-prediction-XGBoostModel` produces a group named
`bank-prediction-XGBoostModel-<6 hex chars>`. The suffix is stable -- every later
version of the same MLflow model syncs into that same group, and MLflow version
*N* maps to package version *N* -- but it means the group name is something you
*read*, not something you assume. Step 7.2 hands the resolved name to Lab 5.

**It has no approval status at all.** The sync does not default it to
`PendingManualApproval`; the field is simply absent until someone sets it. Step
5.8 sets it, after the baseline is attached.

The cell also reads the specification back out of the registry and fails loudly if
the environment variables did not survive the sync -- without `SAGEMAKER_PROGRAM`
the container would fall back to its defaults and never load our handler.


In [ ]:
# Register in MLflow; SageMaker mirrors it into the Model Registry.
registered_model_version = mlflow.register_model(
    f'models:/{logged_model.model_id}', mlflow_model_name
)
print(f'Registered in MLflow: {registered_model_version.name} '
      f'v{registered_model_version.version}')

# The sync runs asynchronously and records the Model Package ARN as a tag on the
# MLflow model version -- a much simpler handle than searching the registry.
model_package_arn = None
for _ in range(40):  # ~2 minutes
    _mv = mlflow_client.get_model_version(
        registered_model_version.name, registered_model_version.version
    )
    model_package_arn = _mv.tags.get('sagemaker.model_package_arn')
    if model_package_arn:
        break
    print('.', end='', flush=True)
    time.sleep(3)

if not model_package_arn:
    raise TimeoutError(
        'No SageMaker Model Package was auto-created. Check that the MLflow app is '
        'in AutoModelRegistrationEnabled mode (Step 2.1) and that its role may call '
        'CreateModelPackage.')

_pkg = sm_client.describe_model_package(ModelPackageName=model_package_arn)

# The auto-created group carries a SageMaker-generated suffix, so read the name
# rather than assuming it. It is stable across versions of the same MLflow model.
mpg_name = _pkg['ModelPackageGroupName']
model_package_version = _pkg['ModelPackageVersion']

print(f'\nSynced SageMaker Model Package')
print(f'  group      : {mpg_name}')
print(f'  version    : {model_package_version}')
print(f'  ARN        : {model_package_arn}')
print(f'  status     : {_pkg["ModelPackageStatus"]}')
print(f'  approval   : {_pkg.get("ModelApprovalStatus", "(not set by the sync)")}')

_raw_container = _pkg['InferenceSpecification']['Containers'][0]
print('\nInference specification, as stored in the registry')
print(f'  image             : {_raw_container["Image"]}')
print(f'  model data source : {_raw_container["ModelDataSource"]["S3DataSource"]["S3Uri"]}')
print(f'  environment       : {_raw_container.get("Environment", {})}')

# # Without SAGEMAKER_PROGRAM the container ignores our handler and serves bare
# # probabilities with no capture -- a silent regression, so fail loudly.
# _expected_env = inference_spec['Containers'][0]['Environment']
# _synced_env = _raw_container.get('Environment', {})
# _mismatched = {k: (v, _synced_env.get(k)) for k, v in _expected_env.items()
#                if _synced_env.get(k) != v}
# if _mismatched:
#     raise RuntimeError(
#         f'Environment did not round-trip through the MLflow sync: {_mismatched}. '
#         'The endpoint would fall back to the container defaults.')
# print('\nEnvironment round-tripped through the sync intact.')


### Step 5.9: Attach `baseline.json` to the package, then approve

`baseline.json` is the artifact that carries this model's reference point
forward. It holds the snapshot IDs, table names, feature schema, evaluation
metrics, and lineage (training job, MLflow run and logged model, code commit).

Four things about it:

**The key names are an API.** Lab 5's `load_baseline_from_registry()` reads these
exact names. Renaming one does not fail loudly -- the monitor just falls back to
live data.

**Snapshot IDs are stored as strings.** They are 19-digit int64 values that lose
precision if a JSON reader parses them as floats. A silently mangled ID is worse
than a missing one.

**It goes in the Glue data bucket, not the default bucket**, because that is what
Lab 5's monitor role is scoped to read. The key includes the training job name,
so each registered version keeps its own immutable baseline.

**The pointer goes on the package as custom metadata.** The natural field would be
`ModelMetrics.ModelQuality.Statistics.S3Uri`, but that can only be set when the
package is *created* -- and here SageMaker created it for us.
`UpdateModelPackage` does accept `CustomerMetadataProperties`, so the baseline URI
and the lineage go there, where Studio also shows them on the version's **Custom
metadata** tab.

**Watch the `approval` line in the output.** `Approved` means both snapshots
pinned and Lab 5 will use this baseline. `PendingManualApproval` means one
snapshot was empty -- deliberate, so an unpinned baseline cannot become the
monitor's fallback. If you see it, revisit Step 3.2 before deploying.


In [ ]:
# Write baseline.json, attach it to the synced Model Package, and approve.
#
# Lab 5 finds the baseline by walking:
#   endpoint -> endpoint config -> model -> Containers[].ModelPackageName
#     -> describe_model_package -> CustomerMetadataProperties['baseline_s3_uri']
#     -> this object -> FROM <training_table> FOR VERSION AS OF <snapshot id>
#
# The key names below are the contract with load_baseline_from_registry() in
# lab5-monitoring/src/drift_monitoring/baseline.py. Renaming one does not fail
# loudly -- the monitor just silently drops back to live data. Snapshot IDs are
# stored as STRINGS: they are 19-digit int64 values that lose precision if a JSON
# reader parses them as floats.
import subprocess

s3 = boto3.client('s3', region_name=region)

# Pin the code that produced this baseline. Studio clones usually are git
# checkouts; when they aren't, the training job name is the next best anchor.
try:
    code_commit_sha = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], stderr=subprocess.DEVNULL, text=True).strip()
except Exception:
    code_commit_sha = training_job_name

baseline = {
    'schema_version': 2,
    'created_at': datetime.now().isoformat(),
    'model_package_group': mpg_name,
    'model_package_arn': model_package_arn,
    'code_commit_sha': code_commit_sha,
    'training_table': TRAINING_TABLE,          # bare table name -- Lab 5 prefixes the database
    'evaluation_table': EVALUATION_TABLE,
    'training_snapshot_id': str(training_snapshot_id),
    'evaluation_snapshot_id': str(evaluation_snapshot_id),
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'feature_schema': FEATURES,
    'metrics': baseline_metrics,
    'sample_size': int(len(y_true)),
    'positive_samples': int(y_true.sum()),
    'negative_samples': int(len(y_true) - y_true.sum()),
    'threshold': 0.5,
    'training_job_name': training_job_name,
    'mlflow_run_id': mlflow_run_id,
    'mlflow_model_id': logged_model.model_id,
    'mlflow_artifact_location': logged_model.artifact_location,
}

# Keyed by training job, so each registered model version keeps its own
# immutable baseline. It goes in the workshop data bucket because that is what
# Lab 5's drift-monitor role is scoped to read -- deliberately with no fallback,
# since a baseline the Lambda cannot read is indistinguishable from no baseline.
baseline_key = f'model-baselines/{training_job_name}/baseline.json'
s3.put_object(Bucket=db_bucket, Key=baseline_key,
              Body=json.dumps(baseline, indent=2).encode('utf-8'),
              ContentType='application/json')

baseline_s3_uri = f's3://{db_bucket}/{baseline_key}'
print(f'baseline.json -> {baseline_s3_uri}')
print(json.dumps({k: baseline[k] for k in (
    'training_table', 'training_snapshot_id',
    'evaluation_table', 'evaluation_snapshot_id')}, indent=2))

# Only approve a baseline that is actually pinned. The monitor's fallback lookup
# considers Approved versions only, so approving an unpinned baseline would make
# "measure drift against live data" the answer for every later monitor run.
_pinned = bool(training_snapshot_id and evaluation_snapshot_id)
_approval = 'Approved' if _pinned else 'PendingManualApproval'

# baseline_s3_uri is the link Lab 5 follows; the rest is lineage, duplicated here
# so it is visible on the version's Custom metadata in Studio without opening the
# JSON. Empty values are rejected by the API, so drop them.
_metadata = {k: v for k, v in {
    'baseline_s3_uri': baseline_s3_uri,
    'training_table': TRAINING_TABLE,
    'evaluation_table': EVALUATION_TABLE,
    'training_snapshot_id': str(training_snapshot_id),
    'evaluation_snapshot_id': str(evaluation_snapshot_id),
    'training_job_name': training_job_name,
    'mlflow_run_id': mlflow_run_id,
    'mlflow_model_id': logged_model.model_id,
    'baseline_roc_auc': f"{baseline_metrics['roc_auc']:.4f}",
}.items() if v}

sm_client.update_model_package(
    ModelPackageArn=model_package_arn,
    CustomerMetadataProperties=_metadata,
    ModelApprovalStatus=_approval,
    ApprovalDescription=(
        f'Baseline pinned to {TRAINING_TABLE}@{training_snapshot_id} / '
        f'{EVALUATION_TABLE}@{evaluation_snapshot_id}' if _pinned
        else 'Unpinned baseline - not eligible as a drift reference'),
)

_pkg = sm_client.describe_model_package(ModelPackageName=model_package_arn)
print(f'\n{mpg_name} version {model_package_version}')
print(f'  baseline    : {_pkg["CustomerMetadataProperties"]["baseline_s3_uri"]}')
print(f'  approval    : {_pkg["ModelApprovalStatus"]}')
if not _pinned:
    print('  NOTE        : left PendingManualApproval because the baseline is unpinned.')


### Step 5.10: Deploy the approved package version

**This is the cell that starts billing.** It runs 5-7 minutes and leaves an
`ml.m5.large` running until you delete it.

Deploying from the registry needs no image, script, artifact URI, or environment
at the call site: all of that is already in the package's
`InferenceSpecification`. A `ContainerDefinition` naming only the Model Package
is enough.

The three typed SDK v3 resources map one-to-one onto SageMaker API objects: a
`Model` (what to serve), an `EndpointConfig` (how much hardware), and an
`Endpoint` (the running HTTPS service).

The model is created from `model_package_name`, not `image` + `model_data_url`.
Only the package reference is supplied here, which both preserves the lineage Lab
5 follows and avoids a second, potentially divergent environment definition.

<div class="alert alert-info">
This endpoint has no authentication of its own -- access is controlled entirely
by IAM (<code>sagemaker:InvokeEndpoint</code>) and it is not exposed to the public
internet. Delete it in the clean-up section when you are done.
</div>


In [ ]:
# Deploy using ONLY the registered Model Package. Its inference specification is
# the single source of image, model artifacts, script-mode flags, and capture
# environment -- do not repeat or override them here.
deployed_model = Model.create(
    model_name=model_name,
    containers=[ContainerDefinition(model_package_name=model_package_arn)],
    execution_role_arn=role,
)
print(f'Model created: {model_name}')

# Confirm the package link exists before starting billable endpoint compute.
_created = sm_client.describe_model(ModelName=model_name)
_pkg_on_model = next((item.get('ModelPackageName')
                      for item in _created.get('Containers', [])
                      if item.get('ModelPackageName')), None)
assert _pkg_on_model == model_package_arn, (
    f'Model {model_name} does not carry the expected ModelPackageName. '
    'Nothing is deployed yet; delete the model and inspect Model.create().')
print(f'  from model package: {_pkg_on_model}')

endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[ProductionVariant(
        variant_name='AllTraffic',
        model_name=model_name,
        initial_instance_count=1,
        instance_type=inference_instance_type,
    )],
)

endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)
print(f'Creating endpoint {endpoint_name} (this takes 5-7 minutes)...')

while True:
    _desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    _status = _desc['EndpointStatus']
    print(f'  {time.strftime("%H:%M:%S")} | status={_status}')
    if _status in ('InService', 'Failed'):
        break
    time.sleep(30)

if _status != 'InService':
    raise RuntimeError(f'Endpoint {endpoint_name} ended in {_status}: '
                       f'{_desc.get("FailureReason")}')
print(f'Endpoint deployed: {endpoint_name}')


> Verify the endpoint creation. Go to the SageMaker AI Studio `Deployments -> Endpoints` and see the new Endpoint

---

## Section 6: Test SageMaker AI Endpoint by predictions


### Step 6.1: Call the endpoint

The next three cells prove the deployed handler works end to end.

The handler accepts both content types deliberately. `application/json` with
named features is the primary path and what these cells use; `text/csv` with
positional values is supported so the traffic-generation step in Lab 5A works
against the same endpoint. The first cell prepares the CSV form, the second
invokes with JSON.

The response shape comes from the handler's `output_fn`:

```json
{"predictions": [0], "probabilities": {"yes": [0.0432], "no": [0.9568]}}
```

That structure is the reason for a custom handler -- the stock container would
return a bare probability with no class and no capture.

**What you should see**: five predictions with probabilities, then a comparison
against the true labels. Do not read much into 5 rows; most customers do not
subscribe, so a run of `No` predictions matching `No` labels is the expected
outcome, not evidence of a degenerate model. The real evaluation is the ROC-AUC
from Step 5.5.

Each of these invocations also publishes a capture message, which is what the
next section verifies.

In [ ]:
# Prepare test data in CSV format (XGBoost expects CSV without headers)
test_sample = X_test.iloc[:5]
test_csv = test_sample.to_csv(header=False, index=False)

print('Test data (first 5 samples):')
print(test_csv[:200] + '...')

In [ ]:
# Invoke the endpoint with JSON (named features) via the runtime client.
# The custom handler returns {"predictions": [...], "probabilities": {"yes":[...],"no":[...]}}
# and captures each prediction to the Iceberg inference_responses table.
runtime = boto3.client('sagemaker-runtime', region_name=region)


def invoke_json(record):
    resp = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Accept='application/json',
        Body=json.dumps(record),
    )
    return json.loads(resp['Body'].read().decode('utf-8'))


sample = X_test.iloc[:5]
predictions = []
print('Sample Predictions (JSON):')
for i in range(len(sample)):
    out = invoke_json(sample.iloc[i].to_dict())
    p_yes = out['probabilities']['yes'][0]
    cls = out['predictions'][0]
    predictions.append(p_yes)
    print(f'  Sample {i+1}: {p_yes:.4f} (Class: {"Yes" if cls == 1 else "No"})')


In [ ]:
# Compare with actual labels
actual_labels = y_test.iloc[:5].values
print('\nPrediction vs Actual:')
for i, (pred, actual) in enumerate(zip(predictions, actual_labels)):
    pred_class = 1 if pred > 0.5 else 0
    match = '\u2713' if pred_class == int(actual) else '\u2717'
    print(f'  {match} Sample {i+1}: Predicted={pred_class}, Actual={int(actual)}, Probability={pred:.4f}')


### Verify Inference Capture (Iceberg)

Capture is asynchronous, which is what keeps it off the serving path: the
handler publishes to SQS and returns immediately, and a logger Lambda drains the
queue into the Iceberg `inference_responses` table.

The Lambda's event source batches on **10 messages or 30 seconds**, whichever
comes first. Five predictions do not fill a batch, so the cell waits 45 seconds
for one batching window plus the Athena `INSERT`.

**What you should see**: a count matching the invocations you just made. Zero
rows usually means you are early -- wait and re-run before assuming something is
broken. If capture was disabled in Step 5.8, the cell says so and skips.

In [ ]:
# Verify captured predictions landed in the Iceberg inference table.
# SQS -> Lambda flushes every 10 messages OR 30s, so wait one cycle first.
import time

if capture_config['ENABLE_INFERENCE_CAPTURE'] == 'true':
    print('Waiting 45s for the SQS -> Lambda flush...')
    time.sleep(45)
    verify_q = f"""
    SELECT COUNT(*) AS n,
           MIN(request_timestamp) AS earliest,
           MAX(request_timestamp) AS latest
    FROM {ATHENA_DATABASE}.inference_responses
    WHERE endpoint_name = '{endpoint_name}'
    """
    res = wr.athena.read_sql_query(
        verify_q, database=ATHENA_DATABASE, s3_output=athena_output, ctas_approach=False
    )
    n = int(res['n'].iloc[0]) if not res.empty else 0
    if n > 0:
        print(f"Captured {n} prediction(s) for {endpoint_name}")
        print(f"  Time range: {res['earliest'].iloc[0]} -> {res['latest'].iloc[0]}")
    else:
        print('No rows yet - wait a little longer for the flush, then re-run.')
else:
    print('Inference capture is disabled (no queue resolved) - nothing to verify.')


---

## Section 7: Verify the Drift-Baseline Chain

Section 5 registered the model with a frozen `baseline.json` and deployed the
endpoint from the synced Model Package. Lab 5's drift monitor now has to find its
way back from the running endpoint to those snapshot IDs, over five hops:

```
endpoint -> endpoint config -> model -> ModelPackageName
  -> CustomerMetadataProperties['baseline_s3_uri'] -> baseline.json
  -> FROM training_data FOR VERSION AS OF <snapshot id>
```

Every hop in that chain fails *silently* — a missing snapshot ID just drops the
`FOR VERSION AS OF` clause, the query still succeeds, and drift is quietly
measured against live data. So rather than trust it, the cell below walks the
chain the same way the monitor does and asserts each hop, then issues the
time-travel query itself to prove the pinned snapshot is readable.

The second cell `%store`s the resolved names for Lab 5 to pick up.


### Step 7.1: Walk the chain

This cell is a test, not a step in the build. It re-walks the five hops above
starting from the endpoint name, asserting each one, and then issues the
`FOR VERSION AS OF` query itself so the pinned snapshot is proven *readable*
rather than merely recorded.

The reason it exists: every hop in that chain fails quietly. A missing snapshot
ID does not raise -- it just drops the time-travel clause, the query still
succeeds, and Lab 5 measures drift against live data. Verifying here means you
find that out now, in a notebook you are already looking at, rather than as an
unexplained flat drift metric two labs later.

It also re-checks the two hops that are specific to the MLflow flow: that the
package's model data source really is the MLflow artifact location, and that
`code/inference.py` is present under it in S3.

**What you should see**: numbered lines, each resolving to a real identifier, the
baseline ROC-AUC matching Step 5.5, and a row count from the pinned training
snapshot.

If an assertion fires, run the cleanup cell at the end **before** re-running
Section 5 -- otherwise the endpoint already created keeps billing and the cleanup
cell will only know about the newer one.


In [ ]:
# Walk the chain exactly the way Lab 5's drift monitor does, and confirm every
# hop resolves. This mirrors load_baseline_from_registry() in
# lab5-monitoring/src/drift_monitoring/baseline.py.
#
# If an assertion below fires, run the cleanup cell at the end of the notebook
# BEFORE re-running Section 5 -- otherwise the endpoint created above keeps
# billing and the cleanup cell will only know about the newer one.
_ep = sm_client.describe_endpoint(EndpointName=endpoint_name)
_cfg = sm_client.describe_endpoint_config(EndpointConfigName=_ep['EndpointConfigName'])
_model_name = _cfg['ProductionVariants'][0]['ModelName']
_model = sm_client.describe_model(ModelName=_model_name)

_arn = None
for _c in _model.get('Containers', []) or [_model.get('PrimaryContainer', {})]:
    if _c.get('ModelPackageName'):
        _arn = _c['ModelPackageName']
        break

print(f'1. endpoint            : {endpoint_name}')
print(f'2. endpoint config     : {_ep["EndpointConfigName"]}')
print(f'3. model               : {_model_name}')
print(f'4. model package       : {_arn or "MISSING -- the walk dead-ends here"}')
assert _arn, 'The deployed model carries no ModelPackageName (see the note above before re-running).'

_pkg = sm_client.describe_model_package(ModelPackageName=_arn)

# The baseline pointer lives in CustomerMetadataProperties because the package was
# created by the MLflow sync, and ModelMetrics can only be set at create time.
# ModelMetrics is still read first so a package registered by the older flow (or
# by the seed-code pipeline) resolves too.
_uri = (_pkg.get('ModelMetrics', {}).get('ModelQuality', {}).get('Statistics', {}).get('S3Uri')
        or _pkg.get('CustomerMetadataProperties', {}).get('baseline_s3_uri'))
print(f'5. baseline.json       : {_uri or "MISSING -- no baseline pointer on the package"}')
assert _uri, ('No baseline pointer on the model package: expected '
              "CustomerMetadataProperties['baseline_s3_uri'] (Step 5.8).")

_b, _k = _uri.replace('s3://', '').split('/', 1)
_loaded = json.loads(s3.get_object(Bucket=_b, Key=_k)['Body'].read())
_train_snap = _loaded['training_snapshot_id']
_eval_snap = _loaded['evaluation_snapshot_id']
print(f'6. data-drift baseline : {ATHENA_DATABASE}.{_loaded["training_table"]} '
      f'FOR VERSION AS OF {_train_snap or "(none -- LIVE table)"}')
print(f'7. model-drift baseline: {ATHENA_DATABASE}.{_loaded["evaluation_table"]} '
      f'FOR VERSION AS OF {_eval_snap or "(none -- LIVE table)"}')
print(f'   baseline ROC-AUC    : {_loaded["metrics"]["roc_auc"]:.4f}')

# The MLflow-specific half: the package serves the artifact store directly, so
# confirm the model data source IS that location and that the serving code sits
# under it in S3. A model file without code/inference.py deploys fine and then
# serves bare probabilities with no capture.
_source = _pkg['InferenceSpecification']['Containers'][0]['ModelDataSource']['S3DataSource']
print(f'8. model data source   : {_source["S3Uri"]} ({_source["S3DataType"]}, '
      f'compression {_source["CompressionType"]})')
assert _source['S3Uri'].rstrip('/') == _loaded['mlflow_artifact_location'].rstrip('/'), (
    'The package does not serve the MLflow artifact location recorded in baseline.json.')

_sb, _sp = _source['S3Uri'].replace('s3://', '').split('/', 1)
_keys = [o['Key'] for o in s3.list_objects_v2(Bucket=_sb, Prefix=_sp).get('Contents', [])]
print(f'9. artifacts in S3     : {len(_keys)} object(s)')
assert any(k.endswith('code/inference.py') for k in _keys), (
    f'code/inference.py is not under {_source["S3Uri"]} -- re-run Step 5.4.')
print('   code/inference.py present under the served prefix')

# Hold the snapshot IDs to the same standard as the hops above -- an empty ID is
# what silently turns the whole chain back into a live-table comparison.
if REQUIRE_PINNED_BASELINE:
    assert _train_snap and _eval_snap, (
        'The registered baseline has no pinned snapshot, so Lab 5 would drop '
        'FOR VERSION AS OF and measure drift against the live table.')

# Issue the time-travel query itself, so the pinned snapshot is proven readable
# rather than merely recorded.
if _train_snap:
    _n = wr.athena.read_sql_query(
        f'SELECT COUNT(*) AS n FROM {_loaded["training_table"]} FOR VERSION AS OF {_train_snap}',
        database=ATHENA_DATABASE, s3_output=athena_output, ctas_approach=False,
    )['n'].iloc[0]
    print(f'\nTime-travel check: {int(_n):,} rows readable in the pinned training snapshot')
    print('Chain verified -- Lab 5 will pin the baseline instead of reading live data.')
else:
    print('\nWARNING: no snapshot pinned. Lab 5 will drop FOR VERSION AS OF and')
    print('measure drift against the LIVE table, which can move under it.')


### Step 7.2: Hand off to Lab 5

`%store` persists these names to IPython's store so Lab 5 can pick them up with
`%store -r` instead of you copying strings between notebooks.

`mpg_name` is worth attention: the auto-sync names the model package group
`bank-prediction-XGBoostModel-<suffix>`, so the exact string is something this
notebook resolved rather than chose. Lab 5's fallback lookup matches the group by
that `bank-prediction-XGBoostModel` prefix, so it works either way -- but if you
set `MODEL_PACKAGE_GROUP` anywhere by hand, use the printed name.

The snapshot IDs are deliberately *not* stored. `baseline.json` on the model
package is their single source of truth, and Lab 5 reads them from there --
storing a second copy here would create a way for the two to disagree.


In [ ]:
# Hand the resolved names to Lab 5 (retrieve there with `%store -r`).
# The snapshot IDs are deliberately NOT stored here: baseline.json on the model
# package is their single source of truth, and Lab 5 reads them from there.
%store endpoint_name
%store model_package_arn
%store mpg_name
%store mlflow_model_name
%store baseline_s3_uri

print('\nLab 5 configuration:')
print(f'  ENDPOINT_NAME       = {endpoint_name}')
print(f'  MLFLOW_MODEL_NAME   = {mlflow_model_name}   (MLflow registry)')
print(f'  MODEL_PACKAGE_GROUP = {mpg_name}   (SageMaker registry, auto-named)')
print(f'  ATHENA_DATABASE     = {ATHENA_DATABASE}')
print(f'  baseline.json       = {baseline_s3_uri}')
print('\nThe endpoint walk above is the primary path and needs no configuration.')
print('MODEL_PACKAGE_GROUP only matters for the fallback used before an endpoint')
print('exists; Lab 5 resolves it by the MLflow model name prefix, so the generated')
print('suffix does not need to be configured anywhere.')


> Verify the registration in SageMaker AI Studio: **Models -> Model registry**, and open the group printed above (`bank-prediction-XGBoostModel-<suffix>`). The new version shows *Approved*, its **Custom metadata** carries `baseline_s3_uri` and the Iceberg snapshot IDs, and its model data source points at the MLflow artifact store. The same version is visible in the MLflow UI under **Models**, where `code/inference.py` sits next to the model files.


---
## Clean up (recommended)

The endpoint is the only resource here that bills continuously. Delete it when
you are done, and re-deploy from Section 5 if you need it again.

**Deleted by the cell below**: the endpoint, its config, and the SageMaker Model.

**Deliberately kept**, because Lab 5 and later labs depend on them: the
registered model package and its version, `baseline.json` in S3, the captured
rows in `inference_responses`, the Iceberg tables, the training job history, and
the MLflow run.

> Lab 5 works against a live endpoint. If you are going straight there, leave
> the endpoint running and come back to this cell afterwards.

In [ ]:
# Delete the endpoint (and its config + model) as it is live and incurs charges.
sm_client.delete_endpoint(EndpointName=endpoint_name)
sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm_client.delete_model(ModelName=model_name)
print(f'Deleted endpoint, config, and model for {endpoint_name}')


## Workshop Summary: Traditional ML with Amazon SageMaker
This workshop provided hands-on experience with the end-to-end machine learning workflow using Amazon SageMaker. Participants learned to:
1. Train models using SageMaker's managed training infrastructure with ModelTrainer, leveraging on-demand compute (XGBoost on ml.m5.xlarge instances) without server management
2. Track experiments with the fully managed Amazon SageMaker MLflow App, logging hyperparameters, metrics (accuracy, AUC, F1), and a **logged model** with its own `model_id`
3. **Make an MLflow model deployable** by logging `code/inference.py` into its artifact store and attaching an `InferenceSpecification` with `log_inference_specification()` — no repacking, and the artifact store stays the single source of truth for the model bytes
4. **Register once, in MLflow**, and let `AutoModelRegistrationEnabled` mirror it into the SageMaker AI Model Registry as a deployable Model Package
5. Attach a frozen `baseline.json` — Iceberg snapshot IDs, feature schema and evaluation metrics — to that package as custom metadata, so drift monitoring in Lab 5 compares against the data this model was actually trained on
6. Deploy the approved package version to a fully managed, real-time HTTPS endpoint with the typed SDK v3 resources (`Model`, `EndpointConfig`, `Endpoint`)
7. The workshop utilized the SageMaker Python SDK v3, highlighting its simplified APIs, reduced boilerplate code, and intelligent defaults.

You've successfully completed the Traditional ML with Amazon SageMaker workshop. You now have hands-on experience with:
- SageMaker's core training and deployment capabilities
- MLflow integration for experiment tracking and as the model source of truth
- Real-world ML workflow from data to deployment
- AWS best practices for ML operations

### Next Steps with SageMaker

**Explore More SageMaker Features:**
- Create SageMaker Pipelines for automated retraining
- Automate ML workflows with CI/CD
- Implement model approval workflows

 Thank You! 🎉
